In [1]:
import os
os.chdir("..")
print(os.getcwd())

c:\Users\NASA2004\OneDrive\Documents\transformer-from-scratch


In [ ]:
# the encoder

import torch
import math
import torch.nn as nn
from src.positional_encoding import PositionalEncoding
from src.attention import MultiHeadAttention

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_output, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

In [3]:
# verifying

d_model, num_heads, d_ff = 512, 8, 2048
layer = EncoderLayer(d_model, num_heads, d_ff)

x = torch.randn(2, 10, d_model)
out = layer(x)
print(out.shape)

torch.Size([2, 10, 512])


In [4]:
# full encoder stacking N layers

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

    def forward(self, x, mask=None):
        x = self.dropout(self.pos_encoding(self.embedding(x) * math.sqrt(self.d_model)))

        for layer in self.layers:
            x = layer(x, mask)

        return x

In [5]:
# verifying

vocab_size = 8000
d_model, num_heads, d_ff, num_layers, max_len = 512, 8, 2048, 6, 100

encoder = Encoder(vocab_size, d_model, num_heads, d_ff, num_layers, max_len)

x = torch.randint(0, vocab_size, (2, 10))
out = encoder(x)
print(out.shape)

torch.Size([2, 10, 512])


In [ ]:
# testing the import from src
from src.encoder import Encoder

encoder = Encoder(vocab_size=8000, d_model=512, num_heads=8, d_ff=2048, num_layers=6, max_len=100)
x = torch.randint(0, 8000, (2, 10))
out = encoder(x)
print(out.shape)

torch.Size([2, 10, 512])
